# 13 · Parent-Child Chunking

**Idea:** small chunks retrieve *precisely* but lack context; large chunks carry context but dilute the match. **Parent-child** (small-to-big) indexes small **child** chunks for matching, then returns their larger **parent** passage to the generator — precision *and* context.

> Runs on Ollama (embeds child chunks).

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import parent_child_chunks

text = (
    'RAG combines a retriever with a generator. The retriever fetches relevant '
    'passages from a corpus at query time. The generator conditions on those '
    'passages to produce a grounded answer. Retrieval quality dominates end-to-end '
    'quality, so chunking, hybrid search, and reranking all matter more than prompt '
    'wording alone.'
)
parents, children = parent_child_chunks(text, source='demo', parent_size=30, child_size=10, overlap=2)
print('parents:', len(parents), '| children:', len(children))
print('\nfirst child :', children[0].text, '-> parent', children[0].meta['parent_index'])

The `ParentChildRetriever` handles the mapping: it searches the small children but hands back the parent passage, deduplicated so you never get two children of the same parent.

In [ ]:
from rag_lab import ParentChildRetriever, load_markdown

retriever = ParentChildRetriever(provider='ollama')
for chunk in load_markdown():
    retriever.add(chunk.text, source=f"{chunk.source}#{chunk.index}", parent_size=80, child_size=25, overlap=5)
print('indexed child chunks:', len(retriever))

In [ ]:
query = 'why do small chunks improve retrieval precision?'
for h in retriever.search(query, k=2):
    print(f'{h.score:5.2f}  {h.chunk.source}')
    print('   parent text:', h.chunk.text[:160], '...')
    print()

The match was made on a short child sentence, but the returned `chunk.text` is the **wider parent** — exactly what you want to hand the generator. The sentence-window variant (notebook 23) is the same idea with a single sentence expanded to its neighbors.

**Takeaway:** decouple the unit you *match on* from the unit you *return*. Match small, return big.

Next → `14_multi_hop_retrieval.ipynb`